# Inspect raw NetCDF datasets

Inventory raw files by source (`ccmp`, `glorys`, `oscar`) and inspect one representative file from each folder. Sampling is bounded and source files are opened read-only.

If needed: `%pip install xarray netCDF4`.

In [ ]:
from pathlib import Path
import xarray as xr

def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / 'data/legacy_dataset/raw').is_dir():
            return path
    raise FileNotFoundError('Cannot locate data/legacy_dataset/raw')

REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / 'data' / 'legacy_dataset' / 'raw'
source_dirs = sorted(p for p in RAW_DIR.iterdir() if p.is_dir())
for source in source_dirs:
    files = sorted(source.glob('*.nc'))
    size_gib = sum(p.stat().st_size for p in files) / 1024**3
    print(f'{source.name}: {len(files)} files, {size_gib:.2f} GiB')
    if files:
        print('  first:', files[0].name)
        print('  last: ', files[-1].name)

In [ ]:
def inspect_netcdf(path):
    """Print a NetCDF schema and a small in-memory sample."""
    print(f'\n## {path.relative_to(REPO_ROOT)}')
    with xr.open_dataset(path) as ds:
        print('Dimensions:', dict(ds.sizes))
        print('Variables:')
        for name, var in ds.data_vars.items():
            print(f'  {name}: dims={var.dims}, shape={var.shape}, dtype={var.dtype}')
        print('Coordinates:')
        for name, coord in ds.coords.items():
            print(f'  {name}: dims={coord.dims}, shape={coord.shape}, dtype={coord.dtype}')
        print('Attributes:', dict(ds.attrs))
        if ds.data_vars:
            name = next(iter(ds.data_vars))
            var = ds[name]
            spatial_names = {'lat', 'latitude', 'lon', 'longitude'}
            indexers = {dim: (slice(0, 10) if dim.lower() in spatial_names else 0) for dim in var.dims}
            sample = var.isel(indexers).compute()
            print(f'Sample {name}: dims={sample.dims}, shape={sample.shape}')
            display(sample)

for source in source_dirs:
    files = sorted(source.glob('*.nc'))
    if files:
        inspect_netcdf(files[0])

## Continue exploring

After checking the schemas, select a specific timestamp or region. Keep selections small before calling `.compute()` or plotting because these grids may be large.